In [1]:
import os
import geopandas as gpd
import pandas as pd
import rasterio

from rasterstats import zonal_stats

## 1. Getting VIIRS metrics

In [2]:
data_path = "../../data"
nightlight_path = "../../data/nightlight_viirs"

raster_main_dict = {
    "nightlights.average_viirs.v21_m_500m_s_20220101_20221231_go_epsg4326_v20250904.tif" : {
        "year": 2022,
    },
    "nightlights.average_viirs.v21_m_500m_s_20230101_20231231_go_epsg4326_v20250904.tif" : {
        "year": 2023,
    },
    "nightlights.average_viirs.v21_m_500m_s_20240101_20241231_go_epsg4326_v20250904.tif" : {
        "year": 2024,
    },
}

In [3]:
ph_bounds = gpd.read_file(os.path.join(data_path,"ph_adm3_municities/PH_Adm3_MuniCities.shp.shp"))
ph_bounds = ph_bounds[ph_bounds["geo_level"] == "City"]
ph_bounds.head()

,adm1_psgc,adm2_psgc,adm3_psgc,adm3_en,geo_level,len_crs,area_crs,len_km,area_km2,geometry
4,100000000,102800000,102805000,City of Batac,City,66661,158252391,66,158.0,"POLYGON ((247341.309 2003933.537, 247293.327 2..."
11,100000000,102800000,102812000,City of Laoag,City,53964,110146974,53,110.0,"POLYGON ((248393.247 2016552.78, 248424.831 20..."
28,100000000,102900000,102906000,City of Candon,City,62247,77652664,62,77.0,"POLYGON ((230319.064 1907309.065, 230338.289 1..."
56,100000000,102900000,102934000,City of Vigan,City,25067,24485368,25,24.0,"POLYGON ((221597.447 1945779.016, 221718.087 1..."
70,100000000,103300000,103314000,City of San Fernando,City,54233,99006121,54,99.0,"POLYGON ((225191.401 1841887.86, 225339.01 184..."


In [4]:
def calculate_ph_viirs_stats(
    ph_bounds: gpd.GeoDataFrame,
    raster: str,
    raster_dict: dict
) -> gpd.GeoDataFrame:

    year = raster_dict.get("year")
    pc_name = raster_dict.get("pc_name")

    print(f"Processing {raster}.")
    raster_path = os.path.join(nightlight_path, raster)

    # Get the CRS and NoData of the raster
    with rasterio.open(raster_path) as src:
        raster_crs = src.crs
        raster_nodata_val = src.nodata
        print(f"Raster CRS: {raster_crs}")
        print(f"Metadata NoData: {raster_nodata_val}")
    
        # Re-project your polygons to match the raster
        _ph_bounds = ph_bounds.copy()
        _ph_bounds = _ph_bounds.to_crs(raster_crs)
        
        # Run the Zonal Statistics
        # Calculate the average value for every polygon in ph_bounds
        stats = zonal_stats(
            _ph_bounds, 
            raster_path,
            stats=['mean', 'max', 'std'], # get multiple metrics at once
            nodata=raster_nodata_val,
            all_touched=True  # ensure small/thin municipalities get data
        )
        
        stats_prefix = year if year else pc_name

        # use a fallback for None values to keep the dataframe clean
        ph_bounds[f'mean_nightlight_{stats_prefix}'] = [x['mean'] if x['mean'] is not None else np.nan for x in stats]
        ph_bounds[f'max_nightlight_{stats_prefix}'] = [x['max'] if x['max'] is not None else np.nan for x in stats]
        ph_bounds[f'std_nightlight_{stats_prefix}'] = [x['std'] if x['std'] is not None else np.nan for x in stats]
        
    return ph_bounds

In [5]:
_ph_bounds = ph_bounds.copy()

for raster, raster_dict in raster_main_dict.items():
    _ph_bounds = calculate_ph_viirs_stats(_ph_bounds, raster, raster_dict)

Processing nightlights.average_viirs.v21_m_500m_s_20220101_20221231_go_epsg4326_v20250904.tif.
Raster CRS: EPSG:4326
Metadata NoData: -32768.0
Processing nightlights.average_viirs.v21_m_500m_s_20230101_20231231_go_epsg4326_v20250904.tif.
Raster CRS: EPSG:4326
Metadata NoData: -32768.0
Processing nightlights.average_viirs.v21_m_500m_s_20240101_20241231_go_epsg4326_v20250904.tif.
Raster CRS: EPSG:4326
Metadata NoData: -32768.0


In [6]:
ph_viirs = _ph_bounds[
    ["adm3_en", "adm3_psgc"] +
    [col for col in _ph_bounds.columns if "nightlight" in col]
].reset_index(drop=True)
ph_viirs.head()

,adm3_en,adm3_psgc,mean_nightlight_2022,max_nightlight_2022,std_nightlight_2022,mean_nightlight_2023,max_nightlight_2023,std_nightlight_2023,mean_nightlight_2024,max_nightlight_2024,std_nightlight_2024
0,City of Batac,102805000,0.494240,19.0,1.352149,0.595622,19.0,1.427617,0.637097,22.0,1.485213
1,City of Laoag,102812000,1.673171,19.0,2.186175,1.962602,18.0,2.238662,2.354472,21.0,2.745203
2,City of Candon,102906000,1.060345,10.0,1.300125,1.217672,9.0,1.265435,1.232759,10.0,1.321948
3,City of Vigan,102934000,2.597403,22.0,3.468821,2.779221,20.0,3.133909,3.025974,24.0,3.501295
4,City of San Fernando,103314000,1.754545,24.0,2.891952,1.905455,20.0,2.879105,2.054545,19.0,3.103770


## 2. Formatting to a city-year panel

In [7]:
# Principal components are treated as static variables
ph_viirs_pc_df = ph_viirs[["adm3_en", "adm3_psgc"] + [col for col in ph_viirs.columns if "pc" in col]]
ph_viirs_pc_df.rename(columns={"adm3_en": "city_name", "adm3_psgc": "psgc"}, inplace=True)
ph_viirs_pc_df.head()

,city_name,psgc
0,City of Batac,102805000
1,City of Laoag,102812000
2,City of Candon,102906000
3,City of Vigan,102934000
4,City of San Fernando,103314000


In [8]:
panel_cols = [
    "city_name", 
    "psgc", 
    "year", 
    "annual_mean_nightlight", 
    "annual_max_nightlight", 
    "annual_std_nightlight"
]

ph_viirs_panel_df = pd.DataFrame(columns=panel_cols)

for year in range(2022, 2025):
    annual_viirs_df = ph_viirs[[
        "adm3_en",
        "adm3_psgc",
        f"mean_nightlight_{year}",
        f"max_nightlight_{year}",
        f"std_nightlight_{year}",
    ]]
    annual_viirs_df["year"] = year
    
    annual_viirs_df.rename(columns={
        "adm3_en": "city_name",
        "adm3_psgc": "psgc",
        f"mean_nightlight_{year}": "annual_mean_nightlight",
        f"max_nightlight_{year}": "annual_max_nightlight",
        f"std_nightlight_{year}": "annual_std_nightlight",
    }, inplace=True)
    
    annual_viirs_df = annual_viirs_df[panel_cols]
    ph_viirs_panel_df = pd.concat([ph_viirs_panel_df, annual_viirs_df], axis=0, ignore_index=True)

ph_viirs_panel_df.head()

,city_name,psgc,year,annual_mean_nightlight,annual_max_nightlight,annual_std_nightlight
0,City of Batac,102805000,2022,0.49424,19.0,1.352149
1,City of Laoag,102812000,2022,1.673171,19.0,2.186175
2,City of Candon,102906000,2022,1.060345,10.0,1.300125
3,City of Vigan,102934000,2022,2.597403,22.0,3.468821
4,City of San Fernando,103314000,2022,1.754545,24.0,2.891952


In [9]:
# Validation: PSGC counts in the panel df vs the VIIRS df shape should be equal
ph_viirs_panel_df[["city_name", "psgc"]].nunique(), ph_viirs_pc_df.shape[0]

(city_name    145
 psgc         149
 dtype: int64,
 149)

In [10]:
ph_viirs_nightlight_df = pd.merge(ph_viirs_panel_df, ph_viirs_pc_df, on=["city_name", "psgc"], how="outer", indicator=True)
ph_viirs_nightlight_df 

,city_name,psgc,year,annual_mean_nightlight,annual_max_nightlight,annual_std_nightlight,_merge
0,Batangas City,401005000,2022,4.823449,980.0,35.509509,both
1,Batangas City,401005000,2023,3.483981,113.0,8.24978,both
2,Batangas City,401005000,2024,5.978868,711.0,34.465678,both
3,City of Alaminos,105503000,2022,0.347555,5.0,0.627099,both
4,City of Alaminos,105503000,2023,0.432882,7.0,0.75921,both
...,...,...,...,...,...,...,...
442,Science City of Muñoz,304917000,2023,0.628169,8.0,0.828911,both
443,Science City of Muñoz,304917000,2024,0.670423,7.0,0.828097,both
444,Tuguegarao City,201529000,2022,2.148148,15.0,2.580033,both
445,Tuguegarao City,201529000,2023,2.502222,16.0,2.838556,both


In [11]:
# Validation: each City-PSGC should have both annual and PC data
ph_viirs_nightlight_df[ph_viirs_nightlight_df._merge != "both"]

,city_name,psgc,year,annual_mean_nightlight,annual_max_nightlight,annual_std_nightlight,_merge


In [12]:
ph_viirs_nightlight_df.drop(columns=["_merge"], inplace=True)

In [13]:
output_path = os.path.join(data_path, "urban_annual_nightlights.csv")
ph_viirs_nightlight_df.to_csv(output_path, index=False)